# Metodi Quantitativi per la Finanza  
## Lezione 5 — Notebook docente

### Simulazione di traiettorie e valori attesi condizionati

Questo notebook sviluppa il caso applicativo della Lezione 5:

> Simulare traiettorie mensili di un prezzo finanziario e stimare il payoff terminale atteso condizionato allo stato osservato a metà orizzonte.

Il notebook è pensato come **versione docente completa**. Contiene codice eseguibile completo, controlli numerici, tabelle intermedie, figure, commenti interpretativi e funzioni riutilizzabili.

## 0. Setup del notebook

Il notebook utilizza soltanto:

`numpy`, `pandas`, `matplotlib`.

Gli output vengono salvati in una cartella locale `output/`, con sottocartelle per tabelle e figure.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.4f}".format

OUTPUT_DIR = Path("output")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parametri del caso applicativo

Si considera un'attività finanziaria con prezzo iniziale

$$
S_0=100.
$$

L'orizzonte è di un anno, suddiviso in dodici periodi mensili:

$$
t=0,1,\ldots,12.
$$

Il prezzo è simulato a partire da rendimenti logaritmici mensili:

$$
R_{t+1}=\log\left(\frac{S_{t+1}}{S_t}\right),
\qquad
R_{t+1}\sim \mathcal{N}(\mu\Delta,\sigma^2\Delta).
$$

Il payoff terminale è

$$
H=(S_{12}-K)^+,
\qquad K=100.
$$

La data informativa intermedia è

$$
t^\ast=6.
$$

Per una preview rapida si può porre `M = 20`.  
Per l'output docente stabile si usa `M = 10_000`.

In [ ]:
S0 = 100.0
T = 12
dt = 1 / 12

mu = 0.04
sigma = 0.20

K = 100.0
M = 10_000

t_star = 6
seed = 202605

param_table = pd.DataFrame({
    "Parametro": ["S0", "T", "dt", "mu", "sigma", "K", "M", "t_star", "seed"],
    "Valore": [S0, T, dt, mu, sigma, K, M, t_star, seed],
    "Descrizione": [
        "Prezzo iniziale",
        "Numero di periodi mensili",
        "Passo temporale annualizzato",
        "Drift logaritmico annuale",
        "Volatilità annuale",
        "Strike del payoff",
        "Numero di traiettorie simulate",
        "Data informativa intermedia",
        "Seed per replicabilità"
    ]
})

param_table

In [ ]:
param_table.to_csv(TABLES_DIR / "tab_01_parametri.csv", index=False)

## 2. Simulazione dei rendimenti logaritmici

Per ogni traiettoria simulata \(k=1,\ldots,M\), si generano \(T\) rendimenti logaritmici mensili:

$$
R^{(k)}_1,\ldots,R^{(k)}_T.
$$

L'oggetto Python `R` è una matrice di dimensione

$$
M \times T.
$$

In [ ]:
rng = np.random.default_rng(seed)

R = rng.normal(
    loc=mu * dt,
    scale=sigma * np.sqrt(dt),
    size=(M, T)
)

assert R.shape == (M, T)

R_preview = pd.DataFrame(
    R[:5, :],
    columns=[f"R{t}" for t in range(1, T + 1)]
)

R_preview

### Controllo descrittivo dei rendimenti simulati

$$
\mathbb{E}[R_{t+1}]=\mu\Delta,
\qquad
\operatorname{sd}(R_{t+1})=\sigma\sqrt{\Delta}.
$$

In [ ]:
return_summary = pd.DataFrame({
    "Quantità": [
        "Media teorica mensile",
        "Media empirica complessiva",
        "Dev. std. teorica mensile",
        "Dev. std. empirica complessiva"
    ],
    "Valore": [
        mu * dt,
        R.mean(),
        sigma * np.sqrt(dt),
        R.std(ddof=1)
    ]
})

return_summary

In [ ]:
return_summary.to_csv(TABLES_DIR / "tab_02_descrittive_rendimenti.csv", index=False)

## 3. Costruzione delle traiettorie di prezzo

Il prezzo si ottiene cumulando i rendimenti logaritmici:

$$
S_t^{(k)}
=
S_0
\exp\left(
\sum_{j=1}^{t}R_j^{(k)}
\right).
$$

La matrice `S` ha dimensione

$$
M \times (T+1).
$$

In [ ]:
cum_R = np.cumsum(R, axis=1)
S_without_S0 = S0 * np.exp(cum_R)

S = np.column_stack([
    np.full(M, S0),
    S_without_S0
])

assert S.shape == (M, T + 1)
assert np.allclose(S[:, 0], S0)
assert np.all(S > 0)

S_preview = pd.DataFrame(
    S[:5, :],
    columns=[f"S{t}" for t in range(T + 1)]
)

S_preview

## 4. Ispezione numerica delle traiettorie

Si confrontano tre sezioni temporali:

$$
S_0, \qquad S_6, \qquad S_{12}.
$$

In [ ]:
def descriptive_stats(x):
    return pd.Series({
        "Media": np.mean(x),
        "Dev. std.": np.std(x, ddof=1),
        "q05": np.quantile(x, 0.05),
        "Mediana": np.quantile(x, 0.50),
        "q95": np.quantile(x, 0.95)
    })


price_summary = pd.DataFrame({
    "S0": descriptive_stats(S[:, 0]),
    "S6": descriptive_stats(S[:, t_star]),
    "S12": descriptive_stats(S[:, -1]),
}).T.reset_index().rename(columns={"index": "Variabile"})

price_summary

In [ ]:
price_summary.to_csv(TABLES_DIR / "tab_03_descrittive_prezzi.csv", index=False)

## 5. Visualizzazione delle traiettorie simulate

Si visualizza un sottoinsieme delle traiettorie. Nel caso \(M=10\,000\), rappresentarle tutte renderebbe il grafico poco leggibile.

In [ ]:
n_paths_to_plot = min(50, M)
time_grid = np.arange(T + 1)

plt.figure(figsize=(9, 5))
for k in range(n_paths_to_plot):
    plt.plot(time_grid, S[k, :], linewidth=1)

plt.axhline(K, linestyle="--", linewidth=1)
plt.xlabel("Tempo mensile t")
plt.ylabel("Prezzo simulato $S_t$")
plt.title(f"Traiettorie simulate del prezzo — prime {n_paths_to_plot} traiettorie")
plt.tight_layout()

fig_path = FIGURES_DIR / "fig_01_traiettorie_simulate.png"
plt.savefig(fig_path, dpi=150)
plt.show()

fig_path

## 6. Payoff terminale

Si considera il payoff terminale

$$
H=(S_{12}-K)^+.
$$

In [ ]:
H = np.maximum(S[:, -1] - K, 0)

assert H.shape == (M,)
assert np.all(H >= 0)

payoff_summary = pd.DataFrame({
    "Quantità": [
        "Media payoff",
        "Deviazione standard payoff",
        "Payoff minimo",
        "Mediana payoff",
        "Payoff massimo",
        "Probabilità empirica payoff positivo"
    ],
    "Valore": [
        H.mean(),
        H.std(ddof=1),
        H.min(),
        np.median(H),
        H.max(),
        np.mean(H > 0)
    ]
})

payoff_summary

In [ ]:
payoff_summary.to_csv(TABLES_DIR / "tab_04_payoff_non_condizionato.csv", index=False)

## 7. Valore atteso non condizionato

$$
\widehat{\mathbb{E}}[H]
=
\frac{1}{M}\sum_{k=1}^{M}H^{(k)}.
$$

In [ ]:
EH_hat = H.mean()
EH_hat

## 8. Distribuzione empirica del prezzo terminale

L'istogramma di \(S_{12}\) visualizza la distribuzione terminale simulata. La linea verticale tratteggiata indica lo strike \(K\).

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(S[:, -1], bins=40)
plt.axvline(K, linestyle="--", linewidth=1)
plt.xlabel("Prezzo terminale $S_{12}$")
plt.ylabel("Frequenza")
plt.title("Distribuzione empirica del prezzo terminale")
plt.tight_layout()

fig_path = FIGURES_DIR / "fig_02_distribuzione_S12.png"
plt.savefig(fig_path, dpi=150)
plt.show()

fig_path

## 9. Informazione intermedia e partizione degli scenari

Alla data

$$
t^\ast=6,
$$

si osserva il livello intermedio \(S_6\). L'informazione disponibile è rappresentata dalla partizione:

$$
A_1=\{S_6<95\},
$$

$$
A_2=\{95\leq S_6<105\},
$$

$$
A_3=\{S_6\geq105\}.
$$

Le maschere booleane sono la traduzione computazionale degli eventi informativi.

In [ ]:
S_star = S[:, t_star]

mask_weak = S_star < 95
mask_neutral = (S_star >= 95) & (S_star < 105)
mask_strong = S_star >= 105

assert np.all(mask_weak | mask_neutral | mask_strong)
assert not np.any(mask_weak & mask_neutral)
assert not np.any(mask_weak & mask_strong)
assert not np.any(mask_neutral & mask_strong)

states = [
    ("Debole", r"$S_6 < 95$", mask_weak),
    ("Laterale", r"$95 \leq S_6 < 105$", mask_neutral),
    ("Favorevole", r"$S_6 \geq 105$", mask_strong),
]

## 10. Frequenze, probabilità empiriche e valori attesi condizionati

Per ciascun evento informativo \(A_g\), si calcolano:

$$
M_g=\#\{k:\omega_k\in A_g\},
$$

$$
\widehat{\mathbb{P}}(A_g)=\frac{M_g}{M},
$$

$$
\widehat{\mathbb{E}}[H\mid A_g]
=
\frac{1}{M_g}
\sum_{k:\omega_k\in A_g}H^{(k)}.
$$

In [ ]:
conditional_rows = []

for state_name, event_label, mask in states:
    Mg = int(mask.sum())
    Pg = Mg / M
    EHg = H[mask].mean() if Mg > 0 else np.nan

    conditional_rows.append({
        "Stato intermedio": state_name,
        "Evento": event_label,
        "Numero traiettorie": Mg,
        "Probabilità empirica": Pg,
        "Payoff medio condizionato": EHg
    })

conditional_table = pd.DataFrame(conditional_rows)

assert conditional_table["Numero traiettorie"].sum() == M
assert np.isclose(conditional_table["Probabilità empirica"].sum(), 1.0)

conditional_table

In [ ]:
conditional_table.to_csv(TABLES_DIR / "tab_05_valori_attesi_condizionati.csv", index=False)

## 11. Costruzione della variabile previsione condizionata

Il valore atteso condizionato rispetto alla sigma-algebra generata dalla partizione

$$
\mathcal{G}=\sigma(A_1,A_2,A_3)
$$

è la variabile casuale

$$
\mathbb{E}[H\mid\mathcal{G}]
=
\mathbb{E}[H\mid A_1]\mathbf{1}_{A_1}
+
\mathbb{E}[H\mid A_2]\mathbf{1}_{A_2}
+
\mathbb{E}[H\mid A_3]\mathbf{1}_{A_3}.
$$

In [ ]:
EH_weak = conditional_table.loc[
    conditional_table["Stato intermedio"] == "Debole",
    "Payoff medio condizionato"
].iloc[0]

EH_neutral = conditional_table.loc[
    conditional_table["Stato intermedio"] == "Laterale",
    "Payoff medio condizionato"
].iloc[0]

EH_strong = conditional_table.loc[
    conditional_table["Stato intermedio"] == "Favorevole",
    "Payoff medio condizionato"
].iloc[0]

Y_hat = np.empty(M)

Y_hat[mask_weak] = EH_weak
Y_hat[mask_neutral] = EH_neutral
Y_hat[mask_strong] = EH_strong

assert Y_hat.shape == (M,)
assert not np.any(np.isnan(Y_hat))

Y_hat[:10]

### Preview delle traiettorie con stato informativo e previsione condizionata

In [ ]:
state_labels = np.empty(M, dtype=object)
state_labels[mask_weak] = "Debole"
state_labels[mask_neutral] = "Laterale"
state_labels[mask_strong] = "Favorevole"

paths_summary = pd.DataFrame({
    "traiettoria": np.arange(1, M + 1),
    "S6": S[:, t_star],
    "S12": S[:, -1],
    "H": H,
    "stato_t6": state_labels,
    "Y_hat": Y_hat
})

paths_summary.head(10)

In [ ]:
paths_summary.to_csv(TABLES_DIR / "tab_06_traiettorie_stato_previsione.csv", index=False)

## 12. Verifica della formula del valore atteso totale

$$
\widehat{\mathbb{E}}[H]
=
\sum_{g=1}^{3}
\widehat{\mathbb{P}}(A_g)
\widehat{\mathbb{E}}[H\mid A_g].
$$

In [ ]:
EH_decomposition = (
    conditional_table["Probabilità empirica"]
    * conditional_table["Payoff medio condizionato"]
).sum()

error_total_expectation = EH_hat - EH_decomposition

total_expectation_table = pd.DataFrame({
    "Quantità": [
        r"$\widehat{\mathbb{E}}[H]$",
        r"$\sum_g \widehat{\mathbb{P}}(A_g)\widehat{\mathbb{E}}[H \mid A_g]$",
        "Errore numerico"
    ],
    "Valore": [
        EH_hat,
        EH_decomposition,
        error_total_expectation
    ]
})

assert np.isclose(EH_hat, EH_decomposition)

total_expectation_table

In [ ]:
total_expectation_table.to_csv(TABLES_DIR / "tab_07_decomposizione_valore_atteso.csv", index=False)

## 13. Grafico dei valori attesi condizionati

La linea orizzontale rappresenta la previsione non condizionata. Le barre rappresentano le previsioni condizionate ai tre stati osservati a \(t^\ast=6\).

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    conditional_table["Stato intermedio"],
    conditional_table["Payoff medio condizionato"]
)
plt.axhline(EH_hat, linestyle="--", linewidth=1)
plt.xlabel("Stato osservato a $t^*=6$")
plt.ylabel("Payoff medio")
plt.title("Valori attesi condizionati del payoff")
plt.tight_layout()

fig_path = FIGURES_DIR / "fig_03_valori_attesi_condizionati.png"
plt.savefig(fig_path, dpi=150)
plt.show()

fig_path

## 14. Distribuzione del payoff per stato intermedio

La media condizionata sintetizza distribuzioni del payoff che possono essere molto diverse. Per questo è utile osservare anche statistiche descrittive condizionate.

In [ ]:
payoff_dist_rows = []

for state_name, event_label, mask in states:
    Hg = H[mask]

    payoff_dist_rows.append({
        "Stato": state_name,
        "Media": Hg.mean() if len(Hg) > 0 else np.nan,
        "Mediana": np.median(Hg) if len(Hg) > 0 else np.nan,
        "q75": np.quantile(Hg, 0.75) if len(Hg) > 0 else np.nan,
        "q95": np.quantile(Hg, 0.95) if len(Hg) > 0 else np.nan,
        "Probabilità payoff positivo": np.mean(Hg > 0) if len(Hg) > 0 else np.nan
    })

payoff_distribution_table = pd.DataFrame(payoff_dist_rows)

payoff_distribution_table

In [ ]:
payoff_distribution_table.to_csv(TABLES_DIR / "tab_08_distribuzione_payoff_per_stato.csv", index=False)

In [ ]:
box_data = [H[mask] for _, _, mask in states]
box_labels = [state_name for state_name, _, _ in states]

plt.figure(figsize=(9, 5))
plt.boxplot(box_data, tick_labels=box_labels, showmeans=True)
plt.axhline(EH_hat, linestyle="--", linewidth=1)
plt.xlabel("Stato osservato a $t^*=6$")
plt.ylabel("Payoff terminale $H$")
plt.title("Distribuzione del payoff per stato intermedio")
plt.tight_layout()

fig_path = FIGURES_DIR / "fig_04_payoff_per_stato.png"
plt.savefig(fig_path, dpi=150)
plt.show()

fig_path

## 15. Interpretazione finanziaria guidata

1. Prima di osservare \(S_6\), la previsione del payoff terminale è unica:

$$
\widehat{\mathbb{E}}[H].
$$

2. Dopo l'osservazione dello stato intermedio, la previsione diventa dipendente dall'informazione disponibile:

$$
\widehat{\mathbb{E}}[H\mid A_g],
\qquad g=1,2,3.
$$

3. La previsione non condizionata non coincide necessariamente con nessuno dei tre valori condizionati: è una media ponderata delle previsioni condizionate.

4. Il controllo della decomposizione del valore atteso totale conferma la coerenza tra calcolo probabilistico e implementazione Python.

In [ ]:
interpretation_table = pd.DataFrame({
    "Domanda": [
        "Il payoff medio condizionato cresce passando da stato debole a favorevole?",
        "La probabilità di payoff positivo aumenta con lo stato intermedio?",
        "La previsione non condizionata è una media ponderata delle previsioni condizionate?",
        "La decomposizione del valore atteso totale è verificata numericamente?"
    ],
    "Risposta da discutere": [
        "Verificare l'ordinamento nella tabella dei valori attesi condizionati.",
        "Confrontare la colonna 'Probabilità payoff positivo' nella tabella per stato.",
        "Sì, per costruzione: controllare la tabella di decomposizione.",
        "Sì, salvo arrotondamenti numerici."
    ]
})

interpretation_table

## 16. Analisi di sensibilità rispetto allo strike \(K\)

Si ripete il calcolo per

$$
K\in\{95,100,105\}.
$$

La simulazione delle traiettorie rimane invariata. Cambia solo il payoff:

$$
H_K=(S_{12}-K)^+.
$$

In [ ]:
K_values = [95.0, 100.0, 105.0]

sensitivity_rows = []

for K_sens in K_values:
    H_K = np.maximum(S[:, -1] - K_sens, 0)

    row = {
        "K": K_sens,
        r"$\widehat{\mathbb{E}}[H_K]$": H_K.mean()
    }

    for state_name, _, mask in states:
        row[state_name] = H_K[mask].mean() if mask.sum() > 0 else np.nan

    sensitivity_rows.append(row)

sensitivity_table = pd.DataFrame(sensitivity_rows)

sensitivity_table

In [ ]:
sensitivity_table.to_csv(TABLES_DIR / "tab_09_sensibilita_K.csv", index=False)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    sensitivity_table["K"],
    sensitivity_table[r"$\widehat{\mathbb{E}}[H_K]$"],
    marker="o",
    label="Non condizionato"
)

for state_name in ["Debole", "Laterale", "Favorevole"]:
    plt.plot(
        sensitivity_table["K"],
        sensitivity_table[state_name],
        marker="o",
        label=state_name
    )

plt.xlabel("Strike $K$")
plt.ylabel("Payoff medio")
plt.title("Sensibilità dei payoff medi rispetto allo strike")
plt.legend()
plt.tight_layout()

fig_path = FIGURES_DIR / "fig_05_sensibilita_K.png"
plt.savefig(fig_path, dpi=150)
plt.show()

fig_path

## 17. Funzioni riutilizzabili

La parte precedente ha sviluppato il caso in modo esplicito.  
Per estensioni, versione studenti e sperimentazioni rapide, è utile raccogliere la logica in funzioni.

In [ ]:
def simulate_price_paths(S0, T, dt, mu, sigma, M, seed):
    # Simula rendimenti logaritmici e traiettorie di prezzo.
    rng = np.random.default_rng(seed)

    R = rng.normal(
        loc=mu * dt,
        scale=sigma * np.sqrt(dt),
        size=(M, T)
    )

    cum_R = np.cumsum(R, axis=1)
    S_without_S0 = S0 * np.exp(cum_R)

    S = np.column_stack([
        np.full(M, S0),
        S_without_S0
    ])

    return R, S


def compute_call_payoff(S, K):
    # Calcola H = (S_T - K)^+.
    return np.maximum(S[:, -1] - K, 0)


def build_information_partition(S, t_star, lower=95.0, upper=105.0):
    # Costruisce la partizione informativa basata su S_{t_star}.
    S_star = S[:, t_star]

    mask_weak = S_star < lower
    mask_neutral = (S_star >= lower) & (S_star < upper)
    mask_strong = S_star >= upper

    assert np.all(mask_weak | mask_neutral | mask_strong)
    assert not np.any(mask_weak & mask_neutral)
    assert not np.any(mask_weak & mask_strong)
    assert not np.any(mask_neutral & mask_strong)

    states = [
        ("Debole", rf"$S_{{{t_star}}} < {lower:g}$", mask_weak),
        ("Laterale", rf"${lower:g} \leq S_{{{t_star}}} < {upper:g}$", mask_neutral),
        ("Favorevole", rf"$S_{{{t_star}}} \geq {upper:g}$", mask_strong),
    ]

    return S_star, states


def conditional_summary(H, states):
    # Calcola frequenze, probabilità empiriche e medie condizionate.
    M = len(H)
    rows = []

    for state_name, event_label, mask in states:
        Mg = int(mask.sum())
        Pg = Mg / M
        EHg = H[mask].mean() if Mg > 0 else np.nan

        rows.append({
            "Stato intermedio": state_name,
            "Evento": event_label,
            "Numero traiettorie": Mg,
            "Probabilità empirica": Pg,
            "Payoff medio condizionato": EHg
        })

    table = pd.DataFrame(rows)

    assert table["Numero traiettorie"].sum() == M
    assert np.isclose(table["Probabilità empirica"].sum(), 1.0)

    return table


def build_conditional_prediction(H, states, conditional_table):
    # Costruisce Y_hat = E_hat[H | G], costante su ciascun elemento della partizione.
    M = len(H)
    Y_hat = np.empty(M)

    for state_name, _, mask in states:
        value = conditional_table.loc[
            conditional_table["Stato intermedio"] == state_name,
            "Payoff medio condizionato"
        ].iloc[0]

        Y_hat[mask] = value

    assert Y_hat.shape == (M,)
    assert not np.any(np.isnan(Y_hat))

    return Y_hat


def check_total_expectation(H, conditional_table):
    # Verifica la decomposizione del valore atteso totale.
    EH_hat = H.mean()

    EH_decomposition = (
        conditional_table["Probabilità empirica"]
        * conditional_table["Payoff medio condizionato"]
    ).sum()

    error = EH_hat - EH_decomposition

    table = pd.DataFrame({
        "Quantità": [
            r"$\widehat{\mathbb{E}}[H]$",
            r"$\sum_g \widehat{\mathbb{P}}(A_g)\widehat{\mathbb{E}}[H \mid A_g]$",
            "Errore numerico"
        ],
        "Valore": [
            EH_hat,
            EH_decomposition,
            error
        ]
    })

    assert np.isclose(EH_hat, EH_decomposition)

    return table

## 18. Esecuzione compatta tramite funzioni

La cella seguente riproduce i risultati principali usando le funzioni appena definite.

In [ ]:
R_fun, S_fun = simulate_price_paths(
    S0=S0,
    T=T,
    dt=dt,
    mu=mu,
    sigma=sigma,
    M=M,
    seed=seed
)

H_fun = compute_call_payoff(S_fun, K=K)

S_star_fun, states_fun = build_information_partition(
    S=S_fun,
    t_star=t_star,
    lower=95.0,
    upper=105.0
)

conditional_table_fun = conditional_summary(H_fun, states_fun)
Y_hat_fun = build_conditional_prediction(H_fun, states_fun, conditional_table_fun)
total_expectation_table_fun = check_total_expectation(H_fun, conditional_table_fun)

display(conditional_table_fun)
display(total_expectation_table_fun)

## 19. Output finale del notebook docente

Il notebook ha prodotto:

1. parametri del caso;
2. matrice dei rendimenti logaritmici simulati;
3. matrice delle traiettorie di prezzo;
4. descrittive di \(S_0\), \(S_6\), \(S_{12}\);
5. grafico delle traiettorie;
6. payoff terminale \(H=(S_{12}-K)^+\);
7. stima non condizionata \(\widehat{\mathbb{E}}[H]\);
8. partizione informativa a \(t^\ast=6\);
9. tabella delle probabilità empiriche e dei valori attesi condizionati;
10. variabile \(\widehat{Y}=\widehat{\mathbb{E}}[H\mid\mathcal{G}]\);
11. verifica della formula del valore atteso totale;
12. grafico delle medie condizionate;
13. distribuzione del payoff per stato;
14. sensibilità rispetto allo strike \(K\);
15. funzioni riutilizzabili per estensioni.

In [ ]:
print("Notebook docente completato.")
print(f"Tabelle salvate in: {TABLES_DIR.resolve()}")
print(f"Figure salvate in:  {FIGURES_DIR.resolve()}")